# SRGAN + binary classifiers (A vs B)

- **Task**: NORMAL vs pathology (DME ∪ DRUSEN), **128×128** inputs for classifiers.
- **Model A**: ResNet18 transfer learning on **real** HR images (train split).
- **SRGAN**: train on **32×32** LR ↔ **128×128** HR pairs from the train split.
- **Model B**: same backbone, trained on **SRGAN-generated** 128×128 train images (≥150 epochs).
- **Test**: both models evaluated on the **same real** 30% hold-out (metrics: accuracy, F1, ROC-AUC).

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
import torchvision.transforms as T

ROOT = Path(".").resolve()
DATA_RAW = ROOT / "data_raw" / "Srinivasan"
SPLIT = ROOT / "outputs" / "split.pt"
SRGAN_CKPT = ROOT / "outputs" / "srgan" / "srgan.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Normalization and augmentation (train vs eval)
ImageNet mean/std on **128×128**; random flips, rotation, color jitter on training only.

In [ ]:
from data_utils import (
    BINARY_NAMES,
    collect_image_paths,
    eval_transform_classifier,
    train_transform_classifier,
    pil_rgb,
)

paths, labels = collect_image_paths(ROOT / "data_raw")
split = torch.load(SPLIT, map_location="cpu")
train_idx = split["train_idx"][:5]

tf_train = train_transform_classifier()
tf_eval = eval_transform_classifier()

fig, axes = plt.subplots(len(train_idx), 4, figsize=(12, 3 * len(train_idx)))
if len(train_idx) == 1:
    axes = np.array([axes])
for row, i in enumerate(train_idx):
    p = paths[i]
    y = labels[i]
    pil = pil_rgb(p)
    axes[row, 0].imshow(np.asarray(pil.resize((128, 128))))
    axes[row, 0].set_title(f"Original {p.name}\n{BINARY_NAMES[y]}")
    axes[row, 0].axis("off")
    for k in range(3):
        torch.manual_seed(42 + k + row * 10)
        aug = tf_train(pil)
        im = aug.permute(1, 2, 0).numpy()
        im = np.clip(im * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)
        axes[row, k + 1].imshow(im)
        axes[row, k + 1].set_title(f"Train aug #{k+1}")
        axes[row, k + 1].axis("off")
plt.tight_layout()
plt.show()

## 2. Super-resolution examples (32×32 → 128×128)
Loads `outputs/srgan/srgan.pt` if present; otherwise shows bicubic upsample vs HR.

In [ ]:
import torch.nn.functional as F

from data_utils import SRGANPairDataset, gan_tensor_to_minus1_1, minus1_1_to_gan_tensor
from models_srgan import SRGANGenerator

ds = SRGANPairDataset(paths, split["train_idx"][:8])
lr, hr = ds[0]
lr_b = lr.unsqueeze(0)
bic = F.interpolate(lr_b, size=(128, 128), mode="bicubic", align_corners=False).squeeze(0)

sr = bic
if SRGAN_CKPT.is_file():
    G = SRGANGenerator(n_residual_blocks=8).to(device)
    ck = torch.load(SRGAN_CKPT, map_location=device)
    G.load_state_dict(ck["G"])
    G.eval()
    with torch.no_grad():
        sr = minus1_1_to_gan_tensor(G(gan_tensor_to_minus1_1(lr_b.to(device)))).cpu().squeeze(0)

fig, ax = plt.subplots(1, 4, figsize=(14, 4))
ax[0].imshow(lr.permute(1, 2, 0).numpy())
ax[0].set_title("LR 32×32")
ax[1].imshow(bic.permute(1, 2, 0).numpy())
ax[1].set_title("Bicubic 128×128")
ax[2].imshow(sr.permute(1, 2, 0).numpy())
ax[2].set_title("SRGAN 128×128" if SRGAN_CKPT.is_file() else "(train SRGAN first)")
ax[3].imshow(hr.permute(1, 2, 0).numpy())
ax[3].set_title("HR 128×128 (target)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

# Mini-grid: first 4 training pairs
fig2, ax2 = plt.subplots(4, 4, figsize=(12, 12))
for i in range(4):
    lr_i, hr_i = ds[i]
    bic_i = F.interpolate(lr_i.unsqueeze(0), size=(128, 128), mode="bicubic", align_corners=False).squeeze(0)
    ax2[i, 0].imshow(lr_i.permute(1, 2, 0).numpy())
    ax2[i, 0].set_ylabel(f"sample {i}")
    ax2[i, 1].imshow(bic_i.permute(1, 2, 0).numpy())
    ax2[i, 2].imshow(hr_i.permute(1, 2, 0).numpy())
    sr_i = bic_i
    if SRGAN_CKPT.is_file():
        with torch.no_grad():
            sr_i = minus1_1_to_gan_tensor(G(gan_tensor_to_minus1_1(lr_i.unsqueeze(0).to(device)))).cpu().squeeze(0)
    ax2[i, 3].imshow(sr_i.permute(1, 2, 0).numpy())
for j, t in enumerate(["LR 32", "Bicubic", "HR", "SRGAN"]):
    ax2[0, j].set_title(t)
for r in range(4):
    for c in range(4):
        ax2[r, c].axis("off")
plt.tight_layout()
plt.show()

## 3. Compare model A vs model B (test set)
Runs the same metric code as `compare_models.py` if checkpoints exist.

In [ ]:
from train_classifier import evaluate_metrics
from torch.utils.data import DataLoader
from data_utils import ImagePathDataset
from models_classifier import build_resnet18_binary

CKPT_A = ROOT / "outputs" / "model_a" / "best.pt"
CKPT_B = ROOT / "outputs" / "model_b" / "best.pt"

paths = [Path(s) for s in split["paths"]]
labels = split["labels"]
test_idx = split["test_idx"]
te_tf = eval_transform_classifier()
test_ds = ImagePathDataset(paths, labels, test_idx, transform=te_tf)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

for name, ck in [("A", CKPT_A), ("B", CKPT_B)]:
    if not ck.is_file():
        print(f"{name}: missing {ck}")
        continue
    m = build_resnet18_binary(num_classes=2, pretrained=False)
    m.load_state_dict(torch.load(ck, map_location=device))
    m.to(device)
    acc, f1, auc, _, _ = evaluate_metrics(m, test_loader, device)
    print(f"Model {name}: accuracy={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}")